# Session 3 — Multi-Agent System (LangGraph)

**GOAL:** Break the single monolithic agent into a TEAM of specialised agents that route work to each other.

```
┌──────────┐  meeting  ┌────────────┐
│  TRIAGE  │──────────>│ SCHEDULER  │
│  AGENT   │           └────────────┘
│          │  task     ┌────────────┐  ┌──────────┐
│          │──────────>│  DRAFTER   │─>│  HUMAN   │
│          │           │   AGENT    │  │  REVIEW  │
└──────────┘           └────────────┘  └──────────┘
     │ info
     v
┌──────────┐
│   END    │
└──────────┘
```

In [ ]:
import os, sys, warnings, logging
from typing import Literal

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from utils.tools import fetch_emails, schedule_meeting, draft_email_reply
from utils.llm_router import get_routed_llm

print('Setup complete.')

### Define Specialised Sub-Agents

Unlike Session 2 where ONE agent had all tools, we now create 3 separate agents.  
Each agent only has the tools and instructions it absolutely needs.

In [ ]:
llm = get_routed_llm(role='master_model')

triage_agent = create_react_agent(
    llm,
    tools=[fetch_emails],
    prompt=SystemMessage(content=(
        'You are the TRIAGE AGENT. Your ONLY job is to:\n'
        '1. Fetch the user\'s recent emails.\n'
        '2. Categorize EACH email as exactly one of: meeting, task, or info.\n'
        '3. After analyzing, respond with a JSON summary.\n'
        'Be precise. Include any time mentioned in meeting emails.'
    )),
)

scheduler_agent = create_react_agent(
    llm,
    tools=[schedule_meeting],
    prompt=SystemMessage(content=(
        'You are the SCHEDULER AGENT. You receive meeting-related emails.\n'
        'Extract the meeting time and attendees, then call schedule_meeting.'
    )),
)

drafter_agent = create_react_agent(
    llm,
    tools=[draft_email_reply],
    prompt=SystemMessage(content=(
        'You are the DRAFTER AGENT. You receive task-related emails.\n'
        'Draft professional, concise replies (under 100 words).'
    )),
)

print('Sub-agents created: Triage, Scheduler, Drafter')

### Define Graph Nodes & Router

In [ ]:
def triage_node(state: MessagesState):
    print('\n[TRIAGE AGENT] Analyzing inbox ...')
    result = triage_agent.invoke(state)
    return {'messages': result['messages']}

def router(state: MessagesState) -> Command[Literal['scheduler_node', 'drafter_node', '__end__']]:
    last_msg = state['messages'][-1]
    content = last_msg.content.lower() if hasattr(last_msg, 'content') else ''
    if 'meeting' in content:
        print('   Route -> SCHEDULER')
        return Command(goto='scheduler_node')
    elif 'task' in content:
        print('   Route -> DRAFTER')
        return Command(goto='drafter_node')
    else:
        print('   Route -> END (only informational)')
        return Command(goto='__end__')

def scheduler_node(state: MessagesState):
    print('\n[SCHEDULER AGENT] Booking ...')
    result = scheduler_agent.invoke({
        'messages': state['messages'] + [HumanMessage(content='Schedule all detected meetings on Google Calendar.')]
    })
    return {'messages': result['messages']}

def drafter_node(state: MessagesState):
    print('\n[DRAFTER AGENT] Drafting ...')
    result = drafter_agent.invoke({
        'messages': state['messages'] + [HumanMessage(content='Draft professional replies for all task-related emails.')]
    })
    return {'messages': result['messages']}

def human_review_node(state: MessagesState):
    print('\n[HUMAN REVIEW] The agent wants to send the following:')
    last_msg = state['messages'][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    print(content[:800])
    human_decision = interrupt('Do you approve this action? (yes/no): ')
    if human_decision.lower().strip() in ('yes', 'y'):
        return {'messages': state['messages'] + [AIMessage(content='Human approved. Action finalized.')]}
    else:
        return {'messages': state['messages'] + [AIMessage(content='Human rejected this action. Discarding.')]}

print('Graph nodes defined.')

### Build & Run the Graph (First Run — will pause at Human Review)

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node('triage_node', triage_node)
graph.add_node('router', router)
graph.add_node('scheduler_node', scheduler_node)
graph.add_node('drafter_node', drafter_node)
graph.add_node('human_review_node', human_review_node)

graph.add_edge(START, 'triage_node')
graph.add_edge('triage_node', 'router')
graph.add_edge('scheduler_node', 'human_review_node')
graph.add_edge('drafter_node', 'human_review_node')
graph.add_edge('human_review_node', END)

app = graph.compile(checkpointer=MemorySaver())
config = {'configurable': {'thread_id': 'hackathon-demo-1'}}

print('Starting multi-agent graph ...\n')
result = app.invoke(
    {'messages': [HumanMessage(content='Analyze my inbox and handle everything appropriately.')]},
    config=config,
)

print('\nAgent conversation (paused for review):')
print('-' * 50)
for msg in result['messages']:
    role = msg.type.upper()
    content = getattr(msg, 'content', '')
    if isinstance(content, list):
        content = '\n'.join(c.get('text', '') if isinstance(c, dict) and 'text' in c else str(c) for c in content)
    if content:
        print(f'\n[{role}]: {content}')

### Resume Graph After Human Input

In [ ]:
human_input = input('HUMAN-IN-THE-LOOP: Do you approve? (yes/no): ').strip()

result = app.invoke(Command(resume=human_input), config=config)

print('\nFINAL RESULT:')
print('-' * 50)
last_msg = result['messages'][-1]
print(f'  {last_msg.content if hasattr(last_msg, "content") else last_msg}')

print('\nKEY TAKEAWAYS:')
print('   1. SPECIALISATION — Each agent has its own tools and prompt.')
print('   2. ROUTING — The graph decides which agent handles what.')
print('   3. HUMAN-IN-THE-LOOP — The graph PAUSED and waited for you!')
print('   4. CHECKPOINTING — If the server crashed, it would resume from the checkpoint.')